### Installing necessary packages and libraries

In [19]:
!pip install sklearn-crfsuite
!pip install python-crfsuite

import sklearn_crfsuite
from sklearn_crfsuite import metrics

from sklearn.model_selection import train_test_split
from collections import Counter

### Loading and Reading the Dataset

In [20]:
filename = "mypos-ver.3.0.shuf.nopipe.txt"
sentences = []

with open(filename, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        sentence = []

        for token in line.split():
            if "/" not in token:
              continue
            word, tag = token.rsplit("/", 1)
            sentence.append((word, tag))

        if sentence:
            sentences.append(sentence)

print("Total sentences:", len(sentences))
print(sentences[:2])

Total sentences: 43196
[[('၁၉၆၂', 'num'), ('ခုနှစ်', 'n'), ('ခန့်မှန်း', 'v'), ('သန်းခေါင်စာရင်း', 'n'), ('အရ', 'ppm'), ('လူဦးရေ', 'n'), ('၁၁၅၉၃၁', 'num'), ('ယောက်', 'part'), ('ရှိ', 'v'), ('သည်', 'ppm'), ('။', 'punc')], [('လူ', 'n'), ('တိုင်း', 'part'), ('တွင်', 'ppm'), ('သင့်မြတ်', 'v'), ('လျော်ကန်', 'v'), ('စွာ', 'part'), ('ကန့်သတ်', 'v'), ('ထား', 'part'), ('သည့်', 'part'), ('အလုပ်', 'n'), ('လုပ်', 'v'), ('ချိန်', 'n'), ('အပြင်', 'conj'), ('၊', 'punc'), ('လစာ', 'n'), ('နှင့်တကွ', 'conj'), ('အခါ', 'n'), ('ကာလ', 'n'), ('အားလျော်စွာ', 'ppm'), ('သတ်မှတ်', 'v'), ('ထား', 'part'), ('သည့်', 'part'), ('အလုပ်', 'n'), ('အားလပ်ရက်', 'n'), ('များ', 'part'), ('ပါဝင်', 'v'), ('သည့်', 'part'), ('အနားယူခွင့်', 'n'), ('နှင့်', 'conj'), ('အားလပ်ခွင့်', 'n'), ('ခံစားပိုင်ခွင့်', 'n'), ('ရှိ', 'v'), ('သည်', 'ppm'), ('။', 'punc')]]


### Feature Extraction

In [21]:
def word2features(sent, i):
    word = sent[i][0]
    features = {
        'bias': 1.0,
        'word.lower': word.lower(),
        'word[-3:]': word[-3:],
        'word[-2:]': word[-2:],
        'word.isupper': word.isupper(),
        'word.istitle': word.istitle(),
        'word.isdigit': word.isdigit(),
        'length': len(word),
    }
    if i > 0:
        prev_word = sent[i-1][0]

        features.update({
            '-1:word.lower': prev_word.lower(),
            '-1:istitle': prev_word.istitle(),
            '-1:isupper': prev_word.isupper(),
        })
    else:
        features['BOS'] = True

    if i < len(sent)-1:
        next_word = sent[i+1][0]

        features.update({
            '+1:word.lower': next_word.lower(),
            '+1:istitle': next_word.istitle(),
            '+1:isupper': next_word.isupper(),
        })
    else:
        features['EOS'] = True
    return features

### Converting Sentences to Features

In [22]:
def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [label for token, label in sent]

def sent2tokens(sent):
    return [token for token, label in sent]

In [23]:
X = [sent2features(s) for s in sentences]
y = [sent2labels(s) for s in sentences]

### Splitting the Dataset into training one and testing one

*Here only one dataset is used. No other dataset is used for testing the model like Sayar's CRF tutorial.*

In [24]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Training CRF Model

In [25]:
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)
crf.fit(X_train, y_train)

CRF(algorithm='lbfgs', all_possible_transitions=True, c1=0.1, c2=0.1,
    max_iterations=100)

### Prediction

In [26]:
y_pred = crf.predict(X_test)

### Evaluation

In [27]:
labels = list(crf.classes_)
labels.remove('punc') if 'punc' in labels else None

print(metrics.flat_classification_report(y_test, y_pred, labels=labels, digits=4))

              precision    recall  f1-score   support

           n     0.9557    0.9699    0.9627     24227
           v     0.9447    0.9382    0.9414     16611
         ppm     0.9802    0.9831    0.9816     17029
         adj     0.8411    0.7992    0.8196      3292
        part     0.9655    0.9670    0.9662     26702
        conj     0.8931    0.9286    0.9105      3474
         int     0.9462    0.8978    0.9213       137
          fw     0.9871    0.8963    0.9395       598
         num     0.9965    0.9796    0.9880      1174
        pron     0.9646    0.9615    0.9630      4021
         adv     0.9086    0.8353    0.8704      2167
          tn     0.9783    0.9599    0.9690      1172
          sb     1.0000    0.7931    0.8846        58
         abb     0.9474    0.7826    0.8571        69

   micro avg     0.9551    0.9551    0.9551    100731
   macro avg     0.9506    0.9066    0.9268    100731
weighted avg     0.9550    0.9551    0.9549    100731



### Overall Accuracy

In [28]:
print("Accuracy:", metrics.flat_accuracy_score(y_test, y_pred))

Accuracy: 0.9593589985024168


### Predicting Tags for a New Sentence

In [35]:
sentence = "ကျွန်တော် မနက်ပိုင်း အစောကြီး ထ ပြီး ကျောင်း မ သွား ချင် ဘူး။ အမေ ရိုက် မှာ ကို ကြောက် တယ်။"
words = sentence.split()

dummy = [(w, "") for w in words]
features = sent2features(dummy)

prediction = crf.predict_single(features)
for w, t in zip(words, prediction):
  print(f"{w:15} {t}")

ကျွန်တော်       pron
မနက်ပိုင်း      n
အစောကြီး        adv
ထ               v
ပြီး            conj
ကျောင်း         n
မ               part
သွား            v
ချင်            part
ဘူး။            adj
အမေ             n
ရိုက်           v
မှာ             ppm
ကို             ppm
ကြောက်          v
တယ်။            n


### Showing most informative features

In [36]:
from collections import Counter
print("Top Positive State Features")

for item in Counter(crf.state_features_).most_common(20):
  print(item)

Top Positive State Features
(('word.lower:ကျွန်တော်', 'pron'), 11.278713)
(('word.lower:ခင်ဗျား', 'pron'), 10.683689)
(('word.lower:လည်းကောင်း', 'conj'), 10.261302)
(('word.lower:ဟင့်အင်း', 'part'), 10.092648)
(('word.lower:ဟင့်အင်း', 'part'), 9.706518)
(('word.lower:တဆိတ်လောက်', 'part'), 9.55047)
(('word.lower:တွေ့တွေ့ချင်း', 'ppm'), 9.524162)
(('word.lower:ကျေးဇူးပြုပြီး', 'int'), 9.467018)
(('word.lower:သော်လည်းကောင်း', 'conj'), 9.374988)
(('word.lower:စသည်', 'part'), 8.999806)
(('word.lower:တုန်း', 'conj'), 8.988671)
(('word.lower:အတွက်', 'ppm'), 8.732297)
(('word.lower:ကျွန်တော့်', 'pron'), 8.661528)
(('word.lower:သိပ်', 'adv'), 8.646215)
(('word.lower:လည်း', 'part'), 8.61815)
(('word.lower:ထို့နောက်', 'conj'), 8.562958)
(('word.lower:ခွင့်', 'n'), 8.47948)
(('word.isupper', 'fw'), 8.277774)
(('word.lower:ထိုစဉ်', 'conj'), 8.272468)
(('word.lower:ဒါမှမဟုတ်', 'conj'), 8.254934)


### Saving the trained model

In [37]:
import joblib
joblib.dump(crf, "myPOS_CRF_model.pkl")

['myPOS_CRF_model.pkl']

In [38]:
crf = joblib.load("myPOS_CRF_model.pkl")